# Text Regression with LASSO

## What it does
Uses LASSO (L1 regularization) to select the **k most predictive text signals** from a high-dimensional
feature matrix (topic proportions, TF-IDF weights, sentence embeddings, etc.) and then runs OLS on the
selected features for interpretable coefficients and inference.

**Two-stage pipeline:**
1. **LASSO path** — scan the regularization path to find the alpha value that leaves exactly `N_SELECT` non-zero coefficients
2. **OLS on selected features** — refit with statsmodels to get standard errors, t-stats, and R²

## When to use it
- You have **many text features** (topics, embedding dimensions) and a macro/financial outcome
- You want an economically interpretable subset of the most predictive signals
- You want to test whether text adds information beyond traditional predictors

## Data format
Two CSV files that share a `date` column:
- **Text features file** — one row per period, one column per text signal (e.g. topic proportions)
- **Outcome file** — one row per period, columns = macro/financial outcomes to predict

The two files are merged on the date column before modelling.

## Configuration

Edit the values below to adapt this notebook to a new dataset.

In [ ]:
CONFIG = {
    # --- Text features file ---
    # Rows = time periods, columns = text signals (topics, TF-IDF, embeddings, ...)
    'TEXT_FILE':       '../../data/topics.csv',   # replace with your file
    'TEXT_DATE_COL':   'date',
    # --- Outcome file ---
    'OUTCOME_FILE':    '../../data/macro.csv',     # replace with your file
    'OUTCOME_DATE_COL':'date',
    # --- Targets to predict ---
    # Single string or list of strings (column names in outcome file)
    'TARGET_COLS':     ['mret'],
    # --- LASSO feature selection ---
    'N_SELECT':        5,        # number of text signals to select
    'N_ALPHAS':        2000,     # resolution of the LASSO path search
    # --- Preprocessing ---
    'STANDARDIZE':     True,
    # --- Output ---
    'SAVE_RESULTS':    True,
    'OUTPUT_DIR':      '../../results',
}

print('Configuration loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Step 1 — Load Data & Merge on Date

In [ ]:
import sys, warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('../../').resolve()))
from utils import load_csv

# Load text features
text_df = load_csv(CONFIG['TEXT_FILE'], parse_dates=[CONFIG['TEXT_DATE_COL']])
text_cols = [c for c in text_df.columns if c != CONFIG['TEXT_DATE_COL']]
print(f'Text features : {text_df.shape[0]} rows × {len(text_cols)} signals')
print(f'Period        : {text_df[CONFIG["TEXT_DATE_COL"]].min().date()} — {text_df[CONFIG["TEXT_DATE_COL"]].max().date()}')

# Load outcomes
outcome_df = load_csv(CONFIG['OUTCOME_FILE'], parse_dates=[CONFIG['OUTCOME_DATE_COL']])
targets = [CONFIG['TARGET_COLS']] if isinstance(CONFIG['TARGET_COLS'], str) else CONFIG['TARGET_COLS']
print(f'\nOutcome file  : {outcome_df.shape[0]} rows × {outcome_df.shape[1]} columns')
print(f'Targets       : {targets}')

In [ ]:
# Align dates to month-start for consistent merging
text_df[CONFIG['TEXT_DATE_COL']]    = text_df[CONFIG['TEXT_DATE_COL']].dt.to_period('M').dt.to_timestamp()
outcome_df[CONFIG['OUTCOME_DATE_COL']] = outcome_df[CONFIG['OUTCOME_DATE_COL']].dt.to_period('M').dt.to_timestamp()

keep_cols = [CONFIG['OUTCOME_DATE_COL']] + targets
df = pd.merge(
    text_df,
    outcome_df[keep_cols],
    left_on  = CONFIG['TEXT_DATE_COL'],
    right_on = CONFIG['OUTCOME_DATE_COL'],
    how      = 'inner',
).drop(columns=[CONFIG['OUTCOME_DATE_COL']], errors='ignore')

print(f'Merged sample : {len(df)} months')
print(f'P/N ratio     : {len(text_cols)}/{len(df)} = {len(text_cols)/len(df):.2f}')
df.head()

## Step 2 — Standardize Features

In [ ]:
X = df[text_cols].values
if CONFIG['STANDARDIZE']:
    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    print('Features standardized.')
else:
    X_scaled = X
    print('Standardization skipped.')

## Step 3 — LASSO Path: Select Top k Features per Target

Scan the LASSO regularization path and find the alpha that yields exactly `N_SELECT` non-zero coefficients.
Then refit OLS on the selected features for interpretable statistics.

In [ ]:
def lasso_select_and_ols(X_scaled, y, feature_names, n_select=5, n_alphas=2000):
    """Run LASSO path, pick alpha giving exactly n_select nonzero features, refit OLS."""
    valid = ~np.isnan(y)
    y_v   = y[valid]
    X_v   = X_scaled[valid]

    alphas_out, coefs_out, _ = lasso_path(X_v, y_v, n_alphas=n_alphas)
    n_nonzero = np.sum(coefs_out != 0, axis=0)

    idx = np.where(n_nonzero == n_select)[0]
    if len(idx) == 0:
        return None

    pick  = idx[0]
    coefs = coefs_out[:, pick]
    nz    = np.where(coefs != 0)[0]
    sel   = [feature_names[j] for j in nz]

    X_ols = sm.add_constant(pd.DataFrame(X_v[:, nz], columns=sel))
    ols   = sm.OLS(y_v, X_ols).fit()

    return {
        'alpha':       alphas_out[pick],
        'selected':    sel,
        'lasso_coefs': {s: coefs[j] for s, j in zip(sel, nz)},
        'ols_model':   ols,
        'n_obs':       int(valid.sum()),
    }

print('Helper defined.')

In [ ]:
all_results = {}

for target in targets:
    print(f'\n{'='*60}')
    print(f'TARGET: {target}')
    print('='*60)

    y = df[target].values
    res = lasso_select_and_ols(X_scaled, y, text_cols,
                               n_select=CONFIG['N_SELECT'], n_alphas=CONFIG['N_ALPHAS'])

    if res is None:
        print(f'  No alpha yielding exactly {CONFIG["N_SELECT"]} non-zero features — try adjusting N_SELECT.')
        continue

    all_results[target] = res
    ols = res['ols_model']

    print(f'  Alpha            : {res["alpha"]:.6f}')
    print(f'  N observations   : {res["n_obs"]}')
    print(f'  Selected features: {res["selected"]}')
    print()
    print(f'  LASSO coefs (standardized):')
    for feat, coef in res['lasso_coefs'].items():
        print(f'    {feat}: {coef:+.6f}')
    print()
    print(f'  OLS R²     : {ols.rsquared:.4f}')
    print(f'  OLS Adj R² : {ols.rsquared_adj:.4f}')
    print()
    print(ols.summary())

## Step 4 — Coefficient Plot per Target

In [ ]:
for target, res in all_results.items():
    ols    = res['ols_model']
    coefs  = ols.params.drop('const')
    conf   = ols.conf_int().drop('const')
    pvals  = ols.pvalues.drop('const')

    fig, ax = plt.subplots(figsize=(8, 4))
    y_pos = range(len(coefs))
    colors = ['steelblue' if v > 0 else 'tomato' for v in coefs]
    ax.barh(y_pos, coefs.values, color=colors, alpha=0.8)
    ax.errorbar(coefs.values, y_pos,
                xerr=[coefs.values - conf[0].values, conf[1].values - coefs.values],
                fmt='none', color='black', capsize=4, linewidth=1.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f'{n} (p={p:.3f})' for n, p in zip(coefs.index, pvals)], fontsize=9)
    ax.invert_yaxis()
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set(xlabel='OLS Coefficient (± 95% CI)',
           title=f'Selected Text Signals → {target}  (R²={ols.rsquared:.3f})')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

## Step 5 — Save Results

In [ ]:
if CONFIG['SAVE_RESULTS']:
    os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

    rows = []
    for target, res in all_results.items():
        ols = res['ols_model']
        rows.append({
            'target':     target,
            'alpha':      res['alpha'],
            'n_selected': len(res['selected']),
            'selected_features': ', '.join(res['selected']),
            'r2':         ols.rsquared,
            'adj_r2':     ols.rsquared_adj,
            'n_obs':      res['n_obs'],
            'standardized': CONFIG['STANDARDIZE'],
            'notebook':   'text_lasso',
        })

    path = os.path.join(CONFIG['OUTPUT_DIR'], 'text_lasso_summary.csv')
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f'Saved: {path}')
else:
    print('SAVE_RESULTS = False — skipping.')